# 01 Data Ingestion And Alignment

In [2]:
# Notebook 01: Data ingestion and alignment

from pathlib import Path
import pandas as pd

from utils.data_utils import (
    align_ccle_with_gdsc,
    load_ccle_gct,
    load_gdsc_response,
    project_root,
    safe_log_ic50,
)

ROOT = project_root()
DATA_DIR = ROOT / "Data"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ccle_path = DATA_DIR / "CCLE_Expression_Entrez_2012-09-29.gct"
# Use the CSV downloaded from the Cell Model Passports GDSC2 section.
gdsc_path = DATA_DIR / "GDSC2_fitted_dose_response_27Oct23.xlsx"

expr_df = load_ccle_gct(ccle_path)
print("CCLE expression shape (genes x samples):", expr_df.shape)

gdsc_df = load_gdsc_response(gdsc_path, drug_name="Erlotinib")
print("Erlotinib rows in GDSC:", gdsc_df.shape[0])

X, y, matches = align_ccle_with_gdsc(expr_df, gdsc_df)
y_log = safe_log_ic50(y)

print("Aligned feature matrix shape:", X.shape)
print("Aligned target shape:", y.shape)

X.to_parquet(OUT_DIR / "X_aligned.parquet")
y.to_frame("ic50").to_parquet(OUT_DIR / "y_aligned.parquet")
y_log.to_frame("log_ic50").to_parquet(OUT_DIR / "y_log_aligned.parquet")
matches.to_csv(OUT_DIR / "cellline_match_table.csv", index=False)

print("Saved aligned artifacts to", OUT_DIR)


CCLE expression shape (genes x samples): (18900, 1037)
Erlotinib rows in GDSC: 955
Aligned feature matrix shape: (617, 18900)
Aligned target shape: (617,)
Saved aligned artifacts to C:\Users\krith\OneDrive - The University of Memphis\Machine Learning project\data\processed


## Data Loading and Alignment — Results

### What we loaded
- **CCLE**: Gene expression matrix with **18,900 genes** across **1,037 cancer cell lines**. 
  Each value represents how actively a gene is expressed in that cell line.
- **GDSC2**: Drug response data filtered to **Erlotinib only**, giving **955 IC50 measurements** 
  across 955 cell lines. IC50 is the regression target, where lower values indicate higher drug sensitivity.

### Alignment step
The two datasets use slightly different naming conventions for cell lines 
(e.g. "MCF7_BREAST" in CCLE vs "MCF7" in GDSC). A name-cleaning function 
(strip punctuation, uppercase, remove tissue suffixes) was applied before matching.

**Result: 617 cell lines appeared in both datasets** — these are the usable samples.

### Final dataset dimensions
| | Value |
|---|---|
| Samples (n) | 617 cell lines |
| Features (p) | 18,900 genes |
| Target (y) | log-transformed IC50 |

### Key observation
This creates a high-dimensional regression problem where the number of features (18,900) is much larger than the number of samples (617). Training ML models directly on raw features would cause overfitting. This motivates the PCA dimensionality reduction step in Notebook 02.

### Artifacts saved
- `X_aligned.parquet` — feature matrix (617 × 18,900)
- `y_aligned.parquet` — raw IC50 values
- `y_log_aligned.parquet` — log-transformed IC50 (used for modeling)
- `cellline_match_table.csv` — record of which cell lines were matched

## References

- Barretina, J., et al. (2012). The Cancer Cell Line Encyclopedia enables 
  predictive modelling of anticancer drug sensitivity. Nature, 483, 603–607.
  https://doi.org/10.1038/nature11003

- Wellcome Sanger Institute. (2023). Genomics of Drug Sensitivity in Cancer 
  (GDSC2) [Dataset]. CancerRxGene. https://www.cancerrxgene.org/